# InferenceGuard: Train ModernBERT Risk Classifier & Host on Hugging Face

This notebook **trains the multi-task ModernBERT-base inferential-risk classifier**
(4 attribute heads: `age`, `location`, `occupation`, `education`) on the committed,
**profile-disjoint SynthPAI splits**, then **pushes the trained model + tokenizer to the
Hugging Face Hub** using `huggingface_hub`.

It **reuses the production training code** in `src/risk_model/train.py`
(`train_risk_model` -> `train_multitask_modernbert`) rather than reimplementing a
training loop, so the notebook stays in sync with the model that `src/risk_model/predict.py`
serves in production.

### Model
The InferenceGuard risk estimator is **`answerdotai/ModernBERT-base`** with per-attribute
attention-pooled classification heads over the project taxonomy
(`src/risk_model/model.py::MultiTaskModernBERTRiskModel`). RoBERTa / DeBERTa only ever
appear as *baseline candidates* in `scripts/benchmark_risk_models.py`; the production risk
classifier is ModernBERT, which is what this notebook trains and hosts.

### Dataset & license
Training uses the SynthPAI dataset (`RobinSta/SynthPAI`), licensed
**CC-BY-NC-SA-4.0 (non-commercial, share-alike)**. Any model trained on SynthPAI
**inherits that non-commercial + share-alike constraint**, and so do derived artifacts
such as our SFT data. See `docs/engineering/production-readiness.md` **section 6
(Data handling controls)** for the license-inheritance analysis before publishing weights.

### Ethics framing
InferenceGuard **reduces inferential-privacy risk, it does not provide anonymity.**
The classifier estimates how *inferable* an author's attributes are from their text so the
rewriter can lower that exposure; it is not a de-anonymization or re-identification tool and
must not be used to profile or identify real individuals.

> **This notebook is not executed in the repository.** It requires a GPU, a Hugging Face
> account, and the SynthPAI download. All code cells are intentionally left unexecuted.

## 1. Environment setup

On Colab, clone the repo and install the training dependencies. Pin versions to match
`requirements-ci.txt` where applicable so the notebook reproduces the CI environment.

On a **Colab T4** note (from `configs/models.yaml` `risk_estimator.t4_notes`): ModernBERT is
designed around FlashAttention-2, which Turing GPUs cannot run, so the backbone is loaded
with `attn_implementation="sdpa"` and `max_length` is capped at 512 (SynthPAI comments are
short, the 8192 context is not needed). **This is already handled inside the model**, so no
action is needed here, it is documented for awareness.

In [ ]:
# If running on Google Colab: clone the repo and cd into it.
# Skip this cell when running locally from inside an existing checkout.
import os

REPO_URL = "https://github.com/AnnaThomas2060/InferenceGuard.git"
REPO_DIR = "InferenceGuard"
# Branch holding the ModernBERT training + hosting changes. These are not yet
# merged into main (PR #1), so clone this branch rather than the default.
REPO_BRANCH = "improvements/modernbert-addition"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not os.path.isdir(REPO_DIR):
    !git clone --branch {REPO_BRANCH} --single-branch {REPO_URL}

if IN_COLAB:
    %cd {REPO_DIR}
    # Ensure we are on the intended branch even if the repo was cloned earlier.
    !git checkout {REPO_BRANCH} && git pull --ff-only origin {REPO_BRANCH}

print("Working directory:", os.getcwd())

In [ ]:
# Install training dependencies.
# Versions should match requirements-ci.txt where applicable (e.g. CPU/GPU torch build).
%pip install -q \
    transformers \
    torch \
    datasets \
    huggingface_hub \
    accelerate \
    scikit-learn \
    pyyaml \
    tqdm

In [ ]:
# Sanity check that the repo's src package is importable from the working directory.
import sys
sys.path.insert(0, os.getcwd())

import torch
from src.risk_model.train import train_risk_model
from src.risk_model.taxonomy import ATTRIBUTE_TAXONOMY

print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
print("taxonomy heads:", list(ATTRIBUTE_TAXONOMY.keys()))

## 2. Data: committed profile-disjoint SynthPAI splits

Training reads the **committed** splits directly:

- `data/splits/train.jsonl` - 210 profiles / 5232 comments
- `data/splits/val.jsonl` - 45 profiles / 1416 comments
- `data/splits/test.jsonl` - 45 profiles / 1175 comments (held out, not used for training)

Counts and SHA-256 hashes are recorded in `data/manifests/split_manifest.json`.

**The splits are profile-disjoint**: every comment from a given author lives in exactly one
split. They must **never be re-split by comment**, doing so would leak an author across
train/val/test and inflate metrics. Use the committed files as-is.

In [ ]:
import json
from pathlib import Path

TRAIN_PATH = "data/splits/train.jsonl"
VAL_PATH = "data/splits/val.jsonl"
MANIFEST_PATH = "data/manifests/split_manifest.json"

# Guarded fallback: only regenerate if the committed splits are missing.
# Regeneration is profile-disjoint (python -m src.data.splits); never re-split by comment.
if not (Path(TRAIN_PATH).exists() and Path(VAL_PATH).exists()):
    print("Committed splits not found; regenerating profile-disjoint splits...")
    !python -m src.data.splits
    !python scripts/check_splits.py
else:
    print("Using committed profile-disjoint splits.")

manifest = json.loads(Path(MANIFEST_PATH).read_text(encoding="utf-8"))
for split in ("train", "val", "test"):
    s = manifest[split]
    print(f"{split:>5}: {s['num_profiles']} profiles / {s['num_comments']} comments")

## 3. Train the multi-task ModernBERT classifier

Call the existing `train_risk_model` entrypoint with `model_type="multitask_modernbert"`.
It dispatches to `train_multitask_modernbert`, which loads the committed splits, builds a
`SynthPAIMultiTaskDataset`, trains `MultiTaskModernBERTRiskModel`, and saves the best
checkpoint (`multitask_heads.pt` + tokenizer) under `artifacts/risk_model/best_model`, plus a
`multitask_training_summary.json` with per-epoch validation accuracy.

We pass the split paths explicitly (`train_data` / `val_data`) so training uses the committed
profile-disjoint files rather than re-deriving a split.

In [ ]:
OUTPUT_DIR = "artifacts/risk_model"

summary = train_risk_model(
    train_data=TRAIN_PATH,
    val_data=VAL_PATH,
    output_dir=OUTPUT_DIR,
    model_name="answerdotai/ModernBERT-base",
    model_type="multitask_modernbert",
    epochs=3,
    batch_size=16,
    lr=3e-5,
)

print("Best macro accuracy:", summary.get("best_macro_accuracy"))
summary

## 4. (Optional) Post-hoc calibration

The **production** calibration path is `src/risk_model/calibrate.py::AttributeTemperatureScaler`
(the same class consumed by `src/risk_model/predict.py::CalibratedRiskPredictor`). It fits a
per-attribute temperature via L-BFGS on **validation** logits and derives Youden's J risk
thresholds.

> Do **not** use `src/risk_model/calibration.py` here, that module is test-only and is not
> wired into any `src/` code.

This cell is lightweight and optional: it collects validation logits from the trained model and
fits the scaler, saving `calibration_params.json` alongside the model artifacts so the hosted
repo can ship a calibrated predictor.

In [ ]:
# Optional: fit the production AttributeTemperatureScaler on validation logits.
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

from src.contracts.schemas import AttributeName
from src.risk_model.calibrate import AttributeTemperatureScaler
from src.risk_model.model import MultiTaskModernBERTRiskModel
from src.risk_model.train import SynthPAIMultiTaskDataset, load_train_val_rows

RUN_CALIBRATION = True  # set False to skip

if RUN_CALIBRATION:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    best_dir = Path(OUTPUT_DIR) / "best_model"

    tokenizer = AutoTokenizer.from_pretrained(best_dir)
    model = MultiTaskModernBERTRiskModel(model_name="answerdotai/ModernBERT-base")
    model.load_state_dict(torch.load(best_dir / "multitask_heads.pt", map_location=device))
    model.to(device).eval()

    _, val_rows = load_train_val_rows(train_data=TRAIN_PATH, val_data=VAL_PATH)
    val_loader = DataLoader(
        SynthPAIMultiTaskDataset(val_rows, tokenizer=tokenizer),
        batch_size=16,
        shuffle=False,
    )

    # Accumulate per-attribute validation logits and labels.
    val_logits = {a.value: [] for a in AttributeName}
    val_labels = {a.value: [] for a in AttributeName}
    with torch.no_grad():
        for batch in val_loader:
            out = model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
            )
            logits_dict = out["logits"] if isinstance(out, dict) else out
            for a in AttributeName:
                val_logits[a.value].append(logits_dict[a.value].detach().cpu())
                val_labels[a.value].append(batch["targets"][a.value].detach().cpu())

    val_logits = {k: torch.cat(v) for k, v in val_logits.items()}
    val_labels = {k: torch.cat(v) for k, v in val_labels.items()}

    scaler = AttributeTemperatureScaler()
    diagnostics = scaler.fit_validation_logits(val_logits, val_labels)
    scaler.save(best_dir / "calibration_params.json")
    print(json.dumps(diagnostics, indent=2))

## 5. Authenticate with the Hugging Face Hub

Log in so the artifacts can be pushed. **Never hardcode a token in the notebook.**
In an interactive session use `notebook_login()`; in a non-interactive/automated run read the
token from the `HF_TOKEN` environment variable instead.

In [ ]:
import os
from huggingface_hub import notebook_login, login, whoami

# Preferred: interactive login (opens a token widget). Never hardcode a token.
if os.environ.get("HF_TOKEN"):
    # Fallback for non-interactive runs: login(token=os.environ['HF_TOKEN'])
    login(token=os.environ["HF_TOKEN"])
else:
    notebook_login()

HF_USER = whoami()["name"]
print("Logged in as:", HF_USER)

## 6. Push the trained model to the Hub

Create the repo (idempotently) and upload the saved artifacts.

> **Note:** `train_risk_model` writes the best checkpoint under `OUTPUT_DIR/best_model`
> (`multitask_heads.pt` + tokenizer, plus the optional `calibration_params.json` from
> Section 4). That `best_model` folder is what gets hosted: we upload its **contents** to the
> Hub repo root, and write the model card (`README.md`) into the same folder so it also lands
> at the repo root. This keeps the hosted layout flat and makes the Section 7 reload
> (`snapshot_download` + `AutoTokenizer.from_pretrained(local_dir)` +
> `torch.load(local_dir/"multitask_heads.pt")`) read from exactly where the files were pushed.

> **Self-describing sidecar:** the cell below also writes a small `multitask_config.json`
> into `best_model/` before uploading. It records the architecture
> (`MultiTaskModernBERTRiskModel`), `base_model`, the attribute order, and
> `num_classes_per_head` (derived from `ATTRIBUTE_TAXONOMY`, so it stays correct if the
> taxonomy changes), making the hosted repo self-describing. This sidecar is **optional**:
> `load_risk_classifier` already works on the existing repo **without** it, inferring
> `answerdotai/ModernBERT-base` and taxonomy-derived head sizes. So re-uploading an
> already-hosted repo just to add the sidecar is not required. The name is
> `multitask_config.json` (not `risk_model_config.json`) so it never collides with the
> `ModernBertRiskClassifier` custom format, which needs both `risk_model_config.json` and
> `model.pt`.

**Substitute your own Hugging Face username / repo name** below, the default derives the repo
id from the logged-in user.

In [ ]:
from huggingface_hub import create_repo, upload_folder
from src.risk_model.taxonomy import ATTRIBUTE_TAXONOMY

# User-substitutable repo id derived from the logged-in account.
repo_id = f"{HF_USER}/inferenceguard-modernbert-risk"

# train_risk_model saves the best checkpoint + tokenizer under OUTPUT_DIR/best_model.
# Host that folder's CONTENTS at the Hub repo root so the Section 7 reload matches.
MODEL_DIR = os.path.join(OUTPUT_DIR, "best_model")

MODEL_CARD = f"""---
license: cc-by-nc-sa-4.0
library_name: transformers
tags:
  - inferential-privacy
  - text-classification
  - modernbert
---

# InferenceGuard ModernBERT Risk Classifier

Multi-task **ModernBERT-base** classifier that estimates how *inferable* an author's
attributes are from short text. It powers the risk-estimation stage of
[InferenceGuard](https://github.com/AnnaThomas2060/InferenceGuard).

## Task
Four attention-pooled classification heads over the project taxonomy:
`age`, `location`, `occupation`, `education`.

## Training data
Trained on the **profile-disjoint** SynthPAI splits committed to the repo
(`data/splits/`): 210 train / 45 val / 45 test profiles. Splits are disjoint by author
profile and must never be re-split by comment.

## Intended use
Estimating inferential-privacy risk so downstream rewriting can reduce attribute exposure.
Consumed in production by `src.risk_model.predict.CalibratedRiskPredictor`.

## Ethics
This model **reduces inference risk, it does not provide anonymity.** It is not a
de-anonymization / re-identification tool and must not be used to profile or identify real
individuals.

## License
Derived from SynthPAI (`RobinSta/SynthPAI`, **CC-BY-NC-SA-4.0**). This model **inherits the
non-commercial + share-alike constraint**; see `docs/engineering/production-readiness.md`
section 6.
"""

create_repo(repo_id, repo_type="model", exist_ok=True)

# Write the model card into best_model/ so it uploads to the repo root alongside the weights.
Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)
Path(MODEL_DIR, "README.md").write_text(MODEL_CARD, encoding="utf-8")

# Write a self-describing architecture sidecar so the hosted repo documents its own
# layout. The loader (src.risk_model.model.load_risk_classifier) reads this file name
# exactly ('multitask_config.json') on the MultiTask branch to override base_model when
# present; it also works WITHOUT this file (defaults answerdotai/ModernBERT-base and
# sizes heads from ATTRIBUTE_TAXONOMY), so re-uploading an existing repo is optional.
# num_classes_per_head is derived from ATTRIBUTE_TAXONOMY so it stays correct if the
# taxonomy changes. The name 'multitask_config.json' is deliberately NOT
# 'risk_model_config.json': the latter is reserved for the ModernBertRiskClassifier
# custom format, which the loader only takes when BOTH risk_model_config.json AND
# model.pt are present (this repo has neither of those two together).
multitask_config = {
    "architecture": "MultiTaskModernBERTRiskModel",
    "base_model": "answerdotai/ModernBERT-base",
    "attributes": [attr.value for attr in ATTRIBUTE_TAXONOMY],
    "num_classes_per_head": {
        attr.value: len(labels) for attr, labels in ATTRIBUTE_TAXONOMY.items()
    },
    "weights_file": "multitask_heads.pt",
    "calibration_file": "calibration_params.json",
}
with open(Path(MODEL_DIR, "multitask_config.json"), "w", encoding="utf-8") as f:
    json.dump(multitask_config, f, indent=2)

# Upload the best_model/ contents to the Hub repo root (flat layout).
upload_folder(
    folder_path=MODEL_DIR,
    repo_id=repo_id,
    repo_type="model",
    commit_message="Upload InferenceGuard ModernBERT risk classifier",
)

print("Pushed to:", f"https://huggingface.co/{repo_id}")

## 7. Load the hosted model back for inference

After hosting, the artifacts can be pulled back and wrapped in the production predictor.
`src/risk_model/predict.py::CalibratedRiskPredictor` is the production consumer: it combines
the ModernBERT logits with the `AttributeTemperatureScaler` to return calibrated per-attribute
risk bands.

In [ ]:
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

from src.risk_model.model import MultiTaskModernBERTRiskModel
from src.risk_model.calibrate import AttributeTemperatureScaler
from src.risk_model.predict import CalibratedRiskPredictor

local_dir = snapshot_download(repo_id=repo_id, repo_type="model")

tokenizer = AutoTokenizer.from_pretrained(local_dir)
model = MultiTaskModernBERTRiskModel(model_name="answerdotai/ModernBERT-base")
model.load_state_dict(torch.load(Path(local_dir) / "multitask_heads.pt", map_location="cpu"))

cal_path = Path(local_dir) / "calibration_params.json"
scaler = (
    AttributeTemperatureScaler.load(cal_path)
    if cal_path.exists()
    else AttributeTemperatureScaler()
)

predictor = CalibratedRiskPredictor(model=model, tokenizer=tokenizer, scaler=scaler)
report = predictor.predict("Just moved to Seattle for a new software engineering role!")
print(report)

---

**Reminder:** this notebook is not executed in the repository. Run it on a GPU runtime with
a Hugging Face account and the SynthPAI dataset available. All code cells above are left
unexecuted on purpose, do not commit fabricated outputs.